# 03. 이상 탐지 모델 비교 (F1)

| 항목 | 내용 |
| --- | --- |
| 바꿀 곳 | 1번 셀의 `DATA_FILE`, `ZONE` 두 줄 (공통 움직임 컬럼은 `USE_COMMON`) |
| 입력 | `data/processed/cal01_base_v1.csv` ~ `cal01_base_v6.csv` 중 하나 |
| 출력 | `data/processed/model_results/` 안의 CSV 3개 (실행할 때마다 덮어씀) |

**LABEL 만드는 규칙** (5번 셀 한 줄)

| DOMAIN_LABEL | STAT_LABEL | LABEL |
| --- | --- | --- |
| 1 | 상관없음 | 1 (팀이 이상으로 확정) |
| 0 | 1 | 0 (팀이 검토해서 해제) |
| 빈칸 | 1 | 1 (통계 후보) |
| 빈칸 | 0 | 0 |
| 빈칸 | 빈칸 | 평가에서 뺌 |

**비교하는 4가지**

| 이름 | 하는 일 | 왜 보나 |
| --- | --- | --- |
| Dummy | 전부 0으로 예측 | 바닥 점수. F1은 항상 0 |
| Z-score | 같은 LOT에서 **지금까지 본 값만으로** \|Z\| > 3이면 1 | 가장 단순한 규칙 |
| IsolationForest | 라벨 없이 평소와 다른 점을 찾음 | 정답 없이 어디까지 찾나 |
| RandomForest | 라벨을 배워서 예측 | 팀 판단을 따라 하나 |

- STAT_LABEL은 LOT 전체(미래 값 포함)의 평균으로 만든 것임. 같은 방법으로 맞히면 F1이 1이 되므로 Z-score는 지난 값만 씀

**평가 방법** (시간 순서 유지)

- LOT 번호 순서가 시간 순서임
- 앞의 LOT들로 학습하고, 바로 다음 LOT 하나로 시험함
- 첫 4개 LOT은 학습에만 쓰고, 5번째부터 14번째 LOT까지 10번 시험함
- 10번 시험한 결과를 모두 모아 F1을 계산함

## 1. 공통 경로와 설정 (`DATA_FILE`, `ZONE` 두 줄만 바꿈)

In [ ]:
import os
import pandas as pd

# 저장소 루트 찾기: notebooks 폴더가 안 보이면 한 칸 위로 (최대 두 번)
ROOT = os.getcwd()
for step in range(2):
    if not os.path.isdir(os.path.join(ROOT, "notebooks")):
        ROOT = os.path.dirname(ROOT)

# 폴더
DATA_DIR = os.path.join(ROOT, "data")                    # 원본 CSV 8개
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")             # data/raw에 두었으면 그 폴더를 씀
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")  # 만든 CSV (cal01_base_v1 ~ v6)
LABEL_DIR = os.path.join(ROOT, "data", "labels")         # 팀 검토 표 (domain_decisions.csv)
FIG_DIR = os.path.join(ROOT, "figures")                  # 그림
ENC = "utf-8-sig"

# 원본 CSV
TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")

# 읽을 가공 CSV (이름만 바꾸면 됨)
DATA_FILE = "cal01_base_v6.csv"  # cal01_base_v1.csv ~ cal01_base_v6.csv
BASE_CSV = os.path.join(PROCESSED_DIR, DATA_FILE)

# 비교할 구역
ZONE = "Z4"  # CLN, Z1 ~ Z9, COOLING

# 공통 움직임 컬럼 (Z3 ~ Z6 온도가 최근 5분 사이에 같이 움직였는지)을 넣을지
USE_COMMON = False

## 2. 라이브러리와 저장 폴더

In [ ]:
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import IsolationForest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RESULT_DIR = os.path.join(PROCESSED_DIR, "model_results")
os.makedirs(RESULT_DIR, exist_ok=True)

# v4 ~ v6은 1분 묶음 데이터
MINUTE_DATA = DATA_FILE in ["cal01_base_v4.csv", "cal01_base_v5.csv", "cal01_base_v6.csv"]
print(DATA_FILE, "/ 1분 데이터:", MINUTE_DATA)

## 3. 구역과 센서

In [ ]:
# 구역별 센서 (CP-Z4M은 값이 0.0000000001 수준이라 라벨 계산에 쓰지 않음)
GROUPS = {
    "CLN": ["CLN-IN"],
    "Z1": ["TC-Z1", "OP-Z1"],
    "Z2": ["TC-Z2", "OP-Z2"],
    "Z3": ["TC-Z3", "OP-Z3"],
    "Z4": ["TC-Z4", "OP-Z4", "CP-Z4"],
    "Z5": ["TC-Z5", "OP-Z5"],
    "Z6": ["TC-Z6", "OP-Z6"],
    "Z7": ["TC-Z7", "OP-Z7"],
    "Z8": ["TC-Z8", "OP-Z8"],
    "Z9": ["TC-Z9"],
    "COOLING": ["TC-OUT1", "TC-OUT2"],
}

# 라벨 계산에 쓰는 센서 21개
SENSORS = []
for zone in GROUPS:
    SENSORS = SENSORS + GROUPS[zone]

# CSV에 저장하는 원본 센서 22개 (CP-Z4M은 저장만 하고 계산에는 안 씀)
SAVE_SENSORS = SENSORS + ["CP-Z4M"]

# 라벨 컬럼 22개 (구역마다 STAT, DOMAIN)
LABEL_COLUMNS = []
for zone in GROUPS:
    LABEL_COLUMNS.append(zone + "__STAT_LABEL")
    LABEL_COLUMNS.append(zone + "__DOMAIN_LABEL")

sensors = GROUPS[ZONE]
print(ZONE, "센서:", sensors)

# 공통 움직임 컬럼에 쓰는 온도 4개 (Z3 변화가 2 ~ 5분 뒤 Z4, Z5, Z6로 이어짐)
COMMON_SENSORS = ["TC-Z3", "TC-Z4", "TC-Z5", "TC-Z6"]

## 4. 데이터 읽기 (필요한 컬럼만)

In [ ]:
columns = ["LOT_NO", "MEAS_DT", "EXCLUDED", ZONE + "__STAT_LABEL", ZONE + "__DOMAIN_LABEL"]

for sensor in sensors:
    columns.append(sensor)

    if MINUTE_DATA:
        columns = columns + [sensor + "__MIN", sensor + "__MAX", sensor + "__STD"]

# 공통 움직임 컬럼을 쓰면 Z3 ~ Z6 온도도 읽음
if USE_COMMON:
    for sensor in COMMON_SENSORS:
        if sensor not in columns:
            columns.append(sensor)

df = pd.read_csv(BASE_CSV, encoding=ENC, usecols=columns)
df["MEAS_DT"] = pd.to_datetime(df["MEAS_DT"], errors="raise")
df = df.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)

print(df.shape)

## 5. LABEL 만들기

도메인 판단이 있으면 도메인 값, 없으면 통계 후보 값을 씀. 제외 행과 라벨이 빈 행은 뺌.

In [ ]:
df["LABEL"] = df[ZONE + "__DOMAIN_LABEL"].fillna(df[ZONE + "__STAT_LABEL"])

# 다른 규칙을 쓰려면 위 한 줄만 바꿈
# 예) 팀이 1로 확정한 것만 1로 볼 때
# df["LABEL"] = (df[ZONE + "__DOMAIN_LABEL"] == 1).astype(int).where(df[ZONE + "__STAT_LABEL"].notna())

df = df[(df["EXCLUDED"] == False) & df["LABEL"].notna()].reset_index(drop=True)
df["LABEL"] = df["LABEL"].astype(int)

print("평가할 행:", len(df))
print("LABEL 1 비율:", round(df["LABEL"].mean() * 100, 2), "%")

df.groupby("LOT_NO")["LABEL"].agg(["count", "sum"])

## 6. 입력 변수 만들기

| 변수 | 뜻 | 원기록 (v1 ~ v3) | 1분 (v4 ~ v6) |
| --- | --- | --- | --- |
| 센서 | 현재 값 | O | O (1분 평균) |
| `__DIFF` | 바로 앞 행과의 차이 | O | O |
| `__DEV60` | 직전 60행 평균과의 차이 | O | |
| `__STD60` | 직전 60행 표준편차 | O | |
| `__MIN`, `__MAX`, `__STD` | 1분 안의 최소, 최대, 표준편차 | | O |
| `온도__CHG5` | Z3 ~ Z6 온도의 5분 전 대비 변화 (`USE_COMMON`일 때) | O | O |
| `COMMON__OTHERS` | 내 구역 말고 Z3 ~ Z6 온도 중 최근 5분 안에 \|Z\| > 3이었던 수 (`USE_COMMON`일 때) | O | O |

- 모든 계산은 같은 LOT 안에서, 현재보다 앞선 행만 사용함
- 공통 움직임의 \|Z\|도 7번 Z-score 기준과 같이 지난 값만으로 계산함
- 5분 전 값이 없는 LOT 앞부분은 변화를 0으로 둠 (컬럼을 켜고 꺼도 평가하는 행이 같도록)

In [ ]:
features = []

for sensor in sensors:
    features.append(sensor)
    features.append(sensor + "__DIFF")

    if MINUTE_DATA:
        features = features + [sensor + "__MIN", sensor + "__MAX", sensor + "__STD"]
    else:
        features = features + [sensor + "__DEV60", sensor + "__STD60"]

lots = sorted(df["LOT_NO"].unique())

for lot in lots:
    rows = df["LOT_NO"] == lot

    for sensor in sensors:
        values = df.loc[rows, sensor]
        previous = values.shift(1)  # 바로 앞 행 값

        df.loc[rows, sensor + "__DIFF"] = values - previous

        if not MINUTE_DATA:
            df.loc[rows, sensor + "__DEV60"] = values - previous.rolling(60, min_periods=10).mean()
            df.loc[rows, sensor + "__STD60"] = previous.rolling(60, min_periods=10).std()

# 1분 안에 행이 하나뿐이면 표준편차가 비므로 0으로 채움
if MINUTE_DATA:
    for sensor in sensors:
        df[sensor + "__STD"] = df[sensor + "__STD"].fillna(0)

# 공통 움직임 컬럼: Z3 ~ Z6 온도가 최근 5분 사이에 같이 움직였는지
if USE_COMMON:
    WINDOW = 5 if MINUTE_DATA else 300  # 1분 데이터는 5행, 원기록은 300행 (둘 다 5분)

    for sensor in COMMON_SENSORS:
        features.append(sensor + "__CHG5")
    features.append("COMMON__OTHERS")

    df["COMMON__OTHERS"] = 0

    for lot in lots:
        rows = df["LOT_NO"] == lot

        for sensor in COMMON_SENSORS:
            values = df.loc[rows, sensor]
            previous = values.shift(1)

            # 5분 전 대비 변화
            df.loc[rows, sensor + "__CHG5"] = (values - values.shift(WINDOW)).fillna(0)

            # 내 구역 온도는 세지 않음
            if sensor in sensors:
                continue

            # 지난 값만으로 |Z| > 3이었는지, 최근 5분 안에 한 번이라도
            past_mean = previous.expanding(min_periods=30).mean()
            past_std = previous.expanding(min_periods=30).std(ddof=0)
            past_std = past_std.replace(0, np.nan)

            over = ((values - past_mean).abs() / past_std > 3).astype(int)
            recent = over.rolling(WINDOW, min_periods=1).max()

            df.loc[rows, "COMMON__OTHERS"] = df.loc[rows, "COMMON__OTHERS"] + recent

# LOT 앞부분처럼 값이 빈 행은 뺌
before = len(df)
df = df.dropna(subset=features).reset_index(drop=True)

print("입력 변수:", features)
print("값이 비어 뺀 행:", before - len(df))

## 7. Z-score 기준 (지난 값만 사용)

- 같은 LOT에서 현재 행 이전까지 본 값의 평균과 표준편차로 \|Z\|를 계산함
- `expanding`은 LOT 처음부터 지금까지 (`rolling`은 최근 N개)
- 앞의 30행은 기준이 부족해서 0으로 둠
- 구역 센서 중 하나라도 \|Z\| > 3이면 1

In [ ]:
df["PRED_Z"] = 0

for lot in lots:
    rows = df["LOT_NO"] == lot

    for sensor in sensors:
        values = df.loc[rows, sensor]
        previous = values.shift(1)

        past_mean = previous.expanding(min_periods=30).mean()
        past_std = previous.expanding(min_periods=30).std(ddof=0)
        past_std = past_std.replace(0, np.nan)

        z = (values - past_mean).abs() / past_std

        over_index = z[z > 3].index
        df.loc[over_index, "PRED_Z"] = 1

print("Z-score가 1로 예측한 행:", int(df["PRED_Z"].sum()))

## 8. 시간 순서대로 학습하고 시험하기

| 시험 LOT | 학습 LOT |
| --- | --- |
| 5번째 LOT | 1 ~ 4번째 |
| 6번째 LOT | 1 ~ 5번째 |
| ... | ... |
| 14번째 LOT | 1 ~ 13번째 |

- IsolationForest: 학습 LOT의 LABEL 1 비율만큼을 이상으로 봄 (`contamination`)
- RandomForest: 1이 적어서 `class_weight="balanced"`로 1에 무게를 더 줌, 나무 깊이는 10까지
- 학습 LOT에 1이 하나도 없으면 두 모델은 0으로 둠

In [ ]:
FIRST_TEST = 4  # 앞의 4개 LOT은 학습에만 씀

df["TEST"] = False
df["PRED_DUMMY"] = 0
df["PRED_IF"] = 0
df["PRED_RF"] = 0

for k in range(FIRST_TEST, len(lots)):
    test_lot = lots[k]
    train_lots = lots[:k]

    train = df[df["LOT_NO"].isin(train_lots)]
    test_rows = df["LOT_NO"] == test_lot

    X_train = train[features]
    y_train = train["LABEL"]
    X_test = df.loc[test_rows, features]

    df.loc[test_rows, "TEST"] = True

    # Dummy: 전부 0
    dummy = DummyClassifier(strategy="constant", constant=0)
    dummy.fit(X_train, y_train)
    df.loc[test_rows, "PRED_DUMMY"] = dummy.predict(X_test)

    rate = y_train.mean()

    if rate > 0:
        # IsolationForest: -1(이상) -> 1, 1(정상) -> 0
        iso = IsolationForest(n_estimators=100, contamination=min(rate, 0.5), random_state=42, n_jobs=-1)
        iso.fit(X_train)
        df.loc[test_rows, "PRED_IF"] = (iso.predict(X_test) == -1).astype(int)

        # RandomForest
        rf = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)
        df.loc[test_rows, "PRED_RF"] = rf.predict(X_test)

    print("시험 LOT", test_lot, "/ 학습 LOT", len(train_lots), "개 / 학습 1 비율", round(rate * 100, 2), "%")

## 9. 전체 결과 (시험한 10개 LOT을 모아서)

| 칸 | 뜻 |
| --- | --- |
| TP | 1을 1로 맞힘 |
| FP | 0인데 1로 경보 (오탐) |
| FN | 1인데 놓침 (미탐) |
| TN | 0을 0으로 맞힘 |
| 정확도 | 참고용. 0이 대부분이라 Dummy도 높게 나옴 |

In [ ]:
MODELS = {
    "Dummy (전부 0)": "PRED_DUMMY",
    "Z-score (지난 값)": "PRED_Z",
    "IsolationForest": "PRED_IF",
    "RandomForest": "PRED_RF",
}

# 공통 움직임 컬럼을 켰으면 결과 이름에 표시 (04에서 따로 보임)
data_name = DATA_FILE
if USE_COMMON:
    data_name = DATA_FILE + " + 공통 움직임"

test = df[df["TEST"] == True]
y_true = test["LABEL"]

rows = []

for name in MODELS:
    y_pred = test[MODELS[name]]

    rows.append({
        "DATA_FILE": data_name,
        "ZONE": ZONE,
        "모델": name,
        "시험 행": len(test),
        "LABEL 1": int(y_true.sum()),
        "TP": int(((y_true == 1) & (y_pred == 1)).sum()),
        "FP": int(((y_true == 0) & (y_pred == 1)).sum()),
        "FN": int(((y_true == 1) & (y_pred == 0)).sum()),
        "TN": int(((y_true == 0) & (y_pred == 0)).sum()),
        "정확도": round(accuracy_score(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_true, y_pred, zero_division=0), 4),
    })

summary = pd.DataFrame(rows)
summary

## 10. 시험 LOT별 F1 (LABEL 1이 없는 LOT은 빈칸)

In [ ]:
rows = []

for lot in lots[FIRST_TEST:]:
    part = test[test["LOT_NO"] == lot]
    row = {"LOT_NO": lot, "시험 행": len(part), "LABEL 1": int(part["LABEL"].sum())}

    for name in MODELS:
        if part["LABEL"].sum() > 0:
            row[name] = round(f1_score(part["LABEL"], part[MODELS[name]], zero_division=0), 4)
        else:
            row[name] = np.nan

    rows.append(row)

by_lot = pd.DataFrame(rows)
by_lot

## 11. 저장

| 파일 | 내용 |
| --- | --- |
| `파일명_구역_summary.csv` | 9번 전체 결과 |
| `파일명_구역_by_lot.csv` | 10번 LOT별 F1 |
| `파일명_구역_predictions.csv` | 시험 행의 정답과 네 모델 예측 (오탐, 미탐 찾을 때 사용) |

- 공통 움직임 컬럼을 켜면 이름 끝에 `_common`이 붙음

In [ ]:
name = DATA_FILE.replace(".csv", "") + "_" + ZONE
if USE_COMMON:
    name = name + "_common"

summary.to_csv(os.path.join(RESULT_DIR, name + "_summary.csv"), index=False, encoding=ENC)
by_lot.to_csv(os.path.join(RESULT_DIR, name + "_by_lot.csv"), index=False, encoding=ENC)

prediction_columns = ["LOT_NO", "MEAS_DT", "LABEL", "PRED_DUMMY", "PRED_Z", "PRED_IF", "PRED_RF"]
test[prediction_columns].to_csv(os.path.join(RESULT_DIR, name + "_predictions.csv"), index=False, encoding=ENC)

print("저장:", name)